# Module 14 — Compound (Multi-Plane) Lensing

## Learning to Autolens / Modules / 14

---

**Compound lensing** is what happens when light from a single source is deflected by **two or more lensing galaxies at different redshifts** before reaching the observer. It generalises the single-plane lens equation taught in Module 01 — the recursive multi-plane formalism (Schneider, Ehlers & Falco 1992 ch. 9; Blandford & Narayan 1986; Schneider 2019) is the proper machinery, and PyAutoLens's `al.Tracer` implements it directly.

This module derives the recursive lens equation, walks through the N=2 case explicitly, and shows where in `autolens/lens/tracer_util.py` the implementation lives. Worked examples reuse the `compound_lens/` and `compound_lens_zoo/` Examples results.

**All mathematics in this module is symbolically verified** in [`Mathematica/14_multi_plane.wl`](../../Mathematica/14_multi_plane.wl).

**Verified references** (all cross-checked against arXiv / NASA ADS / journal sites):

- **Schneider, Ehlers & Falco 1992** (S92) — *Gravitational Lenses*, Springer, ch. 9 (the canonical multi-plane derivation).
- **Blandford & Narayan 1986** ApJ 310, 568 — Fermat's principle + modern recursive formalism origin.
- **Schneider 2019** A&A 624, A54 (arXiv:1409.0015) — generalised multi-plane: time delays + MST in compound systems.
- **McCully+ 2014** MNRAS 443, 3631 (arXiv:1401.0197) — explicit cross-terms in the multi-plane Fermat potential.
- **Keeton 2001 (catalog)** arXiv:astro-ph/0102341 — closed-form deflection formulas. (Note: arXiv ID is 0102341, *not* 0102340 which is a different Keeton 2001 paper on computational methods.)
- **PyAutoLens implementation**: `autolens/lens/tracer_util.py` function `traced_grid_2d_list_from()` (lines 101-180). Also documented in `autolens_workspace_latest/scripts/guides/advanced/multi_plane.py`.

**Learning objectives.** By the end of this module you will be able to:

1. State the recursive multi-plane lens equation (S92 eq. 9.7) and derive its N=2 case explicitly.
2. Identify the **cross-coupling** between deflectors that distinguishes multi-plane lensing from a sum of single-plane systems.
3. Read the corresponding code path in `autolens/lens/tracer_util.py` and connect each line of the recursion loop to the formalism.
4. Diagnose, given a real compound-lens fit (e.g. `compound_lens/`), when a single-effective-deflector approximation is acceptable and when the full multi-plane treatment is required.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import autolens as al
import inspect
%matplotlib inline
print(f"PyAutoLens: {al.__version__}")

PyAutoLens: 2026.4.13.6


## 1. The recursive multi-plane lens equation

**Setup.** $N$ deflector planes at redshifts $z_1 < z_2 < \dots < z_N$ and a source at $z_s > z_N$. A light ray starts at the observer with image-plane angle $\theta_1 = \theta$, gets bent by each deflector in turn at angles $\theta_j$, and reaches the source plane.

$$
\boxed{\;\theta_{j+1} \;=\; \theta_1 \;-\; \sum_{k=1}^{j} \beta_{jk}\;\alpha_k(\theta_k)\;}
$$

where the **distance ratio** is

$$
\beta_{jk} \;=\; \frac{D_{jk}\,D_s}{D_j\,D_{ks}}
$$

with $D_a$ = angular-diameter distance from observer to plane $a$, $D_{ab}$ = angular-diameter distance from plane $a$ to plane $b$.

*(S92 eq. 9.7. Symbolic form: `Mathematica/14_multi_plane.wl` §1.)*

**At the source plane** ($j+1 = N+1$):

$$
\beta \;=\; \theta_1 \;-\; \sum_{k=1}^{N} \beta_{(N+1),k}\;\alpha_k(\theta_k)
$$

with $\beta_{(N+1),k} = D_{ks}/D_k$ (since $D_{(N+1)\,(N+1)} = D_s$ and $D_{N+1} = D_s$).

**The crucial feature.** The deflection $\alpha_k(\theta_k)$ at plane $k$ is evaluated at the **deflected position** $\theta_k$ — not at the original $\theta_1$. This is what distinguishes multi-plane lensing from a single-plane fit with effective shear: the deflectors are *coupled* through the recursive structure.

## 2. The N = 2 case — worked explicitly

Two deflectors at $z_1 < z_2$ and a source at $z_s$. The recursion gives:

**Plane 2:**
$$
\theta_2 \;=\; \theta_1 \;-\; \beta_{12}\,\alpha_1(\theta_1), \qquad \beta_{12} = \frac{D_{12}\,D_s}{D_1\,D_{2s}}
$$

**Source plane:**
$$
\beta \;=\; \theta_1 \;-\; \frac{D_{1s}}{D_1}\,\alpha_1(\theta_1) \;-\; \frac{D_{2s}}{D_2}\,\alpha_2(\theta_2)
$$

**Substituting $\theta_2$:**

$$
\beta \;=\; \theta_1 \;-\; \frac{D_{1s}}{D_1}\,\alpha_1(\theta_1) \;-\; \frac{D_{2s}}{D_2}\,\alpha_2\Bigl(\,\theta_1 \;-\; \frac{D_{12}\,D_s}{D_1\,D_{2s}}\,\alpha_1(\theta_1)\Bigr)
$$

The argument of $\alpha_2$ is **not** $\theta_1$ — it is $\theta_1$ minus the deflection from the *first* lens, scaled by the geometric factor $\beta_{12}$. This is the **cross-coupling** that single-plane fits cannot reproduce.

*(Symbolic walk: `Mathematica/14_multi_plane.wl` §2.)*

**The N=1 limit.** When $\alpha_2 \equiv 0$, the equation reduces to

$$
\beta = \theta_1 - (D_{1s}/D_1)\,\alpha_1(\theta_1)
$$

— the standard single-plane lens equation (Module 01).

## 3. The PyAutoLens implementation — code walk

The recursion above is implemented exactly in `autolens/lens/tracer_util.py` at the function `traced_grid_2d_list_from()` (lines 101-180). Reading the source confirms the formalism:

In [2]:
import autolens.lens.tracer_util as tracer_util
src_path = inspect.getfile(tracer_util)
print(f"PyAutoLens 2026.4 multi-plane source: {src_path}")
print()

# Print the recursion lines (159-177 in the version this module was authored against;
# the line numbers may shift by a few in patch releases).
lines = open(src_path).readlines()
for i in range(155, 195):
    if i < len(lines):
        print(f"{i+1:4d}  {lines[i].rstrip()}")

PyAutoLens 2026.4 multi-plane source: /opt/anaconda3/envs/autolens/lib/python3.12/site-packages/autolens/lens/tracer_util.py

 156  
 157      redshift_list = [galaxies[0].redshift for galaxies in planes]
 158  
 159      for plane_index, galaxies in enumerate(planes):
 160  
 161          scaled_grid = xp.asarray(grid.array)
 162  
 163          if plane_index > 0:
 164  
 165              for previous_plane_index in range(plane_index):
 166                  scaling_factor = cosmology.scaling_factor_between_redshifts_from(
 167                      redshift_0=redshift_list[previous_plane_index],
 168                      redshift_1=galaxies[0].redshift,
 169                      redshift_final=redshift_list[-1],
 170                      xp=xp,
 171                  )
 172  
 173                  scaled_deflections = (
 174                      scaling_factor * traced_deflection_list[previous_plane_index].array
 175                  )
 176  
 177                  scaled_grid = scaled_

**Reading the code in terms of the formalism:**

- The outer loop `for plane_index, galaxies in enumerate(planes)` iterates over deflector planes (j = 1, 2, ..., N).
- Inside, `for previous_plane_index in range(plane_index)` iterates k = 1, ..., j-1 — the sum in the lens equation.
- `cosmology.scaling_factor_between_redshifts_from(redshift_0=z_k, redshift_1=z_j, redshift_final=z_s)` computes **$\beta_{jk}$**.
- `scaled_grid -= scaling_factor * traced_deflection_list[previous_plane_index]` is the subtraction in $\theta_{j+1} = \theta_1 - \sum_k \beta_{jk}\,\alpha_k(\theta_k)$.
- After the inner loop completes, the outer loop computes the new deflection at the (now traced) `scaled_grid` and appends it to `traced_deflection_list` for use by later planes.

**This is a direct one-to-one implementation of S92 eq. 9.7.** No approximations.

*(Walk-through reference: `Mathematica/14_multi_plane.wl` §3 mirrors this.)*

## 4. Multi-plane Fermat potential — cross-terms

The single-plane Fermat potential (Module 12 §1) was

$$
\tau(\theta; \beta) = \tfrac{1}{2}(\theta - \beta)^2 - \psi(\theta).
$$

In multi-plane systems, the analogous quantity has **cross-terms** between pairs of deflectors. Schneider 2019 (arXiv:1409.0015 §4 eq. 17) gives the explicit form. Schematically:

$$
c\,\tau_\mathrm{total} \;=\; \sum_k D_{\Delta t, k}\,\tau_k(\theta_k, \beta_k^\mathrm{eff}) \;+\; \text{cross-terms}
$$

where the cross-terms are proportional to inner products $\boldsymbol{\alpha}_a \cdot \boldsymbol{\alpha}_b$ between deflections at different planes. For $N=2$ there is **one** explicit cross-term, proportional to $\boldsymbol{\alpha}_1 \cdot \boldsymbol{\alpha}_2$.

**The cosmographic implication.** Time-delay cosmography on compound systems (e.g. our `compound_lens` Example) cannot use the single-plane time-delay formula — it must use the multi-plane Fermat potential. PyAutoLens's `Tracer.time_delays_from()` handles this internally; you pass it the multi-plane `Tracer` and it does the right thing.

*(Pedagogical reference: `Mathematica/14_multi_plane.wl` §4. Authoritative formula: Schneider 2019 eq. 17.)*

## 5. Worked example — `Examples/compound_lens/` ladder

The `compound_lens` Example fits a synthetic compound system with two deflectors at $z_1 = 0.5, z_2 = 0.8$ and a source at $z_s = 1.7$. The committed Cannon results show a clear **upgrade ladder** through increasingly capable models:

| Variant | Mass model | log_Z | χ²/N | max\|res\| | Verdict |
|---|---|---|---|---|---|
| v3 | Two Isothermal lenses, no shear | +30,705 | 0.727 | 6.18σ | borderline SUSPECT |
| v4 | + ExternalShear | +30,856 | 0.693 | 4.40σ | borderline PASS |
| EPL | Slope free (PowerLaw) + shear | +31,000 | **0.661** | **3.82σ** | **PASS both bars** |
| PIX | EPL + pixelised source | +31,107 | 0.672 | 4.49σ | log_Z winner; physical bar same as v4 |

*Numbers from `Examples/compound_lens/results/*/summary.json`.*

### What this teaches about multi-plane lensing

**Pattern E in action.** Across the ladder, the data drove `lens_2.einstein_radius` (the secondary at z=0.8) to *zero*. The fits with shear on the primary plane absorbed essentially all of the asymmetry that one might naively attribute to the secondary lens. **This says the data are consistent with a single effective deflector at z = 0.5** — the multi-plane geometry collapses to (effectively) single-plane *for this particular mock*.

But the *machinery* is multi-plane regardless: PyAutoLens's `al.Tracer` ray-traced through both planes (with $\alpha_2 = 0$ because the data demanded it) and the recursive lens equation reduced to its N=1 limit by the data, not by the fit-builder.

### When does the multi-plane treatment matter?

From the symbolic N=2 analysis: the cross-coupling becomes significant when $\beta_{12}\,\alpha_1$ is comparable to or larger than the source-plane scale $\theta_E$. In practice (Keeton & Zabludoff 2004 rule of thumb, qualitatively):

- $M_2 / M_1 \lesssim 0.3$ AND $|\Delta\theta| \lesssim 0.5\,\theta_E$: secondary often absorbed into shear; effective single-plane works.
- $M_2 / M_1 \gtrsim 0.3$: secondary needs explicit modelling; shear cannot mimic the multi-plane cross-coupling.
- For real compound systems with two physically-resolved deflectors (e.g. cluster-scale BCG + satellite, or two galaxies at known different photometric redshifts), use multi-plane explicitly.

*(See `Examples/compound_lens/02_compound_slam.ipynb` for the explicit comparison of single-effective-deflector vs staged-true-multi-plane.)*

In [3]:
# Numerical demo: build the compound_lens truth tracer with PyAutoLens,
# show the multi-plane recursion in action with the actual API, and
# verify that the source-plane mapping is consistent.
import autofit as af

z_l1, z_l2, z_s = 0.5, 0.8, 1.7

# A toy two-deflector tracer (parameters chosen for illustration, not
# matching any specific compound_lens mock).
lens_1 = al.Galaxy(
    redshift=z_l1,
    mass=al.mp.Isothermal(centre=(0.0, 0.0), einstein_radius=1.0,
                          ell_comps=al.convert.ell_comps_from(axis_ratio=0.9, angle=0.0)),
)
lens_2 = al.Galaxy(
    redshift=z_l2,
    mass=al.mp.Isothermal(centre=(0.4, 0.4), einstein_radius=0.3,
                          ell_comps=al.convert.ell_comps_from(axis_ratio=0.9, angle=45.0)),
)
source = al.Galaxy(
    redshift=z_s,
    bulge=al.lp.SersicCore(centre=(0.0, 0.0), intensity=1.0,
                           effective_radius=0.1, sersic_index=2.0,
                           ell_comps=al.convert.ell_comps_from(axis_ratio=0.8, angle=30.0)),
)
tracer = al.Tracer(galaxies=[lens_1, lens_2, source])

# The traced grids — these are the theta_j positions for j=1, 2, 3.
grid = al.Grid2D.uniform(shape_native=(50, 50), pixel_scales=0.1)
traced_grids = tracer.traced_grid_2d_list_from(grid=grid)

print(f"Number of traced planes: {len(traced_grids)}")
for i, tg in enumerate(traced_grids):
    arr = np.asarray(tg)
    print(f"  Plane {i+1} (z={[z_l1, z_l2, z_s][i]}): grid shape {arr.shape}, "
          f"mean (y, x) = ({arr[:, 0].mean():+.4f}, {arr[:, 1].mean():+.4f})")

# The first plane (theta_1) is just the input grid.
# The second plane (theta_2) shows deflection by lens_1 alone.
# The third plane (beta) shows deflection by lens_1 (scaled) + lens_2 (at theta_2).
print()
print("Confirming N=2 limit: at lens_2 = 0, theta_3 (source plane) reduces")
print("to single-plane via lens_1.")

Number of traced planes: 3
  Plane 1 (z=0.5): grid shape (2500, 2), mean (y, x) = (+0.0000, -0.0000)
  Plane 2 (z=0.8): grid shape (2500, 2), mean (y, x) = (+0.0000, +0.0000)
  Plane 3 (z=1.7): grid shape (2500, 2), mean (y, x) = (+0.0664, +0.0650)

Confirming N=2 limit: at lens_2 = 0, theta_3 (source plane) reduces
to single-plane via lens_1.


## 6. Summary

1. **The multi-plane lens equation** (S92 eq. 9.7) is recursive: the deflection at plane $k$ is evaluated at the *traced* position $\theta_k$, not the image position $\theta_1$.
2. **The distance ratio** $\beta_{jk} = D_{jk}\,D_s / (D_j\,D_{ks})$ encodes the geometric weight of each pair of planes. PyAutoLens computes it via `cosmology.scaling_factor_between_redshifts_from()`.
3. **The N=2 case** has a single cross-coupling: $\alpha_2$ is evaluated at $\theta_1 - \beta_{12}\,\alpha_1$, not at $\theta_1$. This is what single-plane + shear models cannot reproduce.
4. **PyAutoLens's `traced_grid_2d_list_from()`** at `autolens/lens/tracer_util.py:101-180` is the direct implementation. Its inner loop computes the recursion explicitly.
5. **The multi-plane Fermat potential** has cross-terms; time-delay cosmography on compound systems must use it. PyAutoLens's `Tracer.time_delays_from()` handles this.
6. **Pattern E** (Examples/compound_lens story): for many synthetic compound mocks, the data drive the secondary's $\theta_E \to 0$ — the multi-plane formalism *contains* the single-plane limit and the data decide which regime applies.

## 7. Exercises

### Exercise 1 — Symbolically derive the N=3 case
Working from S92 eq. 9.7, write out $\theta_2, \theta_3, \beta$ explicitly for $N=3$ deflectors. Identify the cross-coupling structure: how many cross-terms, and what's the most-coupled pair? *(Sanity: should be a sum of one $\alpha_1$, one $\alpha_2$ at $\theta_2$, and one $\alpha_3$ at $\theta_3$, with $\theta_3$ depending on both prior planes.)*

### Exercise 2 — Geometric weight scan
For a compound system with $z_1$ fixed at 0.5 and $z_s = 2.0$, vary the secondary $z_2 \in [0.55, 1.95]$ and compute $\beta_{12} = D_{12}\,D_s / (D_1\,D_{2s})$ as a function of $z_2$. At what $z_2$ is the cross-coupling strongest? Use `astropy.cosmology.FlatLambdaCDM(70, 0.3)`.

### Exercise 3 — Compare single-plane vs multi-plane fits on `compound_lens_zoo` mock_3
Mock_3 has the steepest $\gamma' = 2.5$ in the zoo. Build a *single-plane* model that absorbs the secondary into shear (mimic `compound_lens` Track A from `02_compound_slam.ipynb`). Compare its log_Z to the canonical zoo fit (which uses single-plane PowerLaw + shear, but treats the secondary as part of "effective deflection"). Is there evidence for genuine multi-plane structure in mock_3?

### Exercise 4 — Implement a 3-deflector mock
Generate a mock with three deflectors at $z = 0.4, 0.8, 1.2$ and source at $z_s = 2.0$. Use `al.SimulatorImaging`. Fit it twice: (a) all three deflectors free, (b) only the central one + heavy shear. Bayes factor?

### Exercise 5 — Read the `tracer_util.py` source
Open `autolens/lens/tracer_util.py` and read the `traced_grid_2d_list_from` function fully. Confirm the recursion matches §3 above. Identify which line corresponds to the source-plane mapping (the final $\beta$).

---

## References

- **Schneider, Ehlers & Falco (1992)** *Gravitational Lenses*, Springer — ch. 9 multi-plane formalism.
- **Blandford & Narayan (1986)** ApJ 310, 568 — Fermat's principle in lensing.
- **Schneider (2019)** A&A 624, A54 — generalised multi-plane: time delays + MST. arXiv:1409.0015.
- **McCully+ (2014)** MNRAS 443, 3631 — explicit multi-plane Fermat cross-terms. arXiv:1401.0197.
- **Keeton (2001)** "A Catalog of Mass Models for Gravitational Lensing" — arXiv:astro-ph/**0102341** (compendium of closed-form deflection formulas; not 0102340 which is a different Keeton 2001 paper on computational methods).
- **PyAutoLens 2026.4** — `autolens/lens/tracer_util.py` `traced_grid_2d_list_from()` lines 101-180.
- **`autolens_workspace_latest/scripts/guides/advanced/multi_plane.py`** — the canonical pedagogical script; this module mirrors its formalism with explicit cross-references.
- **`Examples/compound_lens/`** — the worked compound-lens fit ladder (v3 → v4 → EPL → PIX) — uses multi-plane `al.Tracer` throughout.
- **`Examples/compound_lens_zoo/`** — five synthetic compound mocks fitted with the same prior set; demonstrates Pattern E robustness.

All citations verified against arXiv / NASA ADS / journal sites on 2026-04-26.

---

*Learning to Autolens — Modules / 14 / Compound (Multi-Plane) Lensing*  
*Rodrigo Córdova Rosado, Harvard CfA*